# Spanish DNI/NIE Validation
Validates Spanish national identity documents using the modulo-23 algorithm.

- **DNI**: 8 digits + 1 control letter (e.g., `12345678Z`)
- **NIE**: X/Y/Z + 7 digits + 1 control letter (e.g., `X1234567C`)

**Configure the cell below**, then **Run All**. The final cell adds validation flags — run it manually.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
TABLE_NAME = "{{TABLE_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
# Columns containing DNI/NIE values
DNI_COLUMNS = {{DNI_COLUMNS}}  # e.g., ["dni", "documento_identidad"]
OUTPUT_SUFFIX = "_cleaned"
# ───────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import *
import re

spark = SparkSession.builder.getOrCreate()
# Read from _cleaned if it exists (previous fixes), else original
try:
    df = spark.table(f"{TABLE_NAME}{OUTPUT_SUFFIX}")
    print(f"Reading from {TABLE_NAME}{OUTPUT_SUFFIX} (previous fixes applied)")
except:
    df = spark.table(TABLE_NAME)
    print(f"Reading from {TABLE_NAME} (original table)")
original_count = df.count()

print(f"Table: {TABLE_NAME}")
print(f"Rows: {original_count:,}")

In [ ]:
# ── DNI/NIE Validation UDF ─────────────────────────────────────
DNI_LETTERS = "TRWAGMYFPDXBNJZSQVHLCKE"

def validate_dni_nie(value):
    """
    Validates a Spanish DNI or NIE number.
    Returns (is_valid: bool, reason: str).

    Algorithm:
    - DNI: number % 23 → lookup letter in TRWAGMYFPDXBNJZSQVHLCKE
    - NIE: replace X=0, Y=1, Z=2, then same mod-23 algorithm
    """
    if value is None or str(value).strip() == "":
        return (False, "empty")

    doc = str(value).strip().upper().replace(" ", "").replace("-", "").replace(".", "")

    # NIE format: X/Y/Z + 7 digits + 1 letter
    nie_match = re.match(r'^([XYZ])(\d{7})([A-Z])$', doc)
    if nie_match:
        prefix_map = {"X": "0", "Y": "1", "Z": "2"}
        full_number = int(prefix_map[nie_match.group(1)] + nie_match.group(2))
        expected = DNI_LETTERS[full_number % 23]
        if nie_match.group(3) == expected:
            return (True, "valid_nie")
        return (False, f"nie_bad_letter_expected_{expected}")

    # DNI format: 8 digits + 1 letter
    dni_match = re.match(r'^(\d{8})([A-Z])$', doc)
    if dni_match:
        number = int(dni_match.group(1))
        expected = DNI_LETTERS[number % 23]
        if dni_match.group(2) == expected:
            return (True, "valid_dni")
        return (False, f"dni_bad_letter_expected_{expected}")

    return (False, "invalid_format")

validate_schema = StructType([
    StructField("is_valid", BooleanType(), False),
    StructField("reason", StringType(), False)
])
validate_dni_udf = udf(validate_dni_nie, validate_schema)

print("DNI/NIE validation UDF registered.")

In [ ]:
print("=" * 60)
print("SPANISH DNI/NIE VALIDATION")
print("=" * 60)

for col_name in DNI_COLUMNS:
    print(f"\n── Column: {col_name} ──")

    non_null_df = df.where(
        F.col(col_name).isNotNull() & (F.trim(F.col(col_name)) != "")
    )
    non_null_count = non_null_df.count()

    if non_null_count == 0:
        print("  All values are null or empty — skipping.")
        continue

    validated = non_null_df.withColumn(
        "_validation", validate_dni_udf(F.col(col_name))
    ).withColumn(
        "_is_valid", F.col("_validation.is_valid")
    ).withColumn(
        "_reason", F.col("_validation.reason")
    )

    valid_count = validated.where(F.col("_is_valid") == True).count()
    invalid_count = non_null_count - valid_count
    valid_pct = round(valid_count / non_null_count * 100, 2)

    print(f"  Total non-null: {non_null_count:,}")
    print(f"  Valid: {valid_count:,} ({valid_pct}%)")
    print(f"  Invalid: {invalid_count:,} ({round(100 - valid_pct, 2)}%)")

    if invalid_count > 0:
        print("\n  Breakdown of invalid values:")
        display(
            validated.where(F.col("_is_valid") == False)
            .groupBy("_reason").count().orderBy(F.desc("count"))
        )
        print("\n  Sample invalid values:")
        display(
            validated.where(F.col("_is_valid") == False)
            .select(col_name, "_reason").limit(20)
        )
    else:
        print("  ✓ All DNI/NIE values are valid!")

---
## ⚠ Apply Fix — Add DNI Validation Flags
The cell below adds boolean `{column}_valid` columns. Invalid rows are **NOT removed**, only flagged.

**Review the analysis above before running this cell.**

In [ ]:
# ⚠ APPLY FIX — Run manually after review
cleaned_df = df

for col_name in DNI_COLUMNS:
    flag_col = f"{col_name}_valid"
    cleaned_df = cleaned_df.withColumn(
        "_validation", validate_dni_udf(F.col(col_name))
    ).withColumn(
        flag_col, F.col("_validation.is_valid")
    ).drop("_validation")

output_table = f"{TABLE_NAME}{OUTPUT_SUFFIX}"
cleaned_df.write.mode("overwrite").format("delta").saveAsTable(
    output_table
)

print(f"Added DNI/NIE validation flags for {len(DNI_COLUMNS)} columns")
print(f"Output: {output_table}")